# P2-05 Parquet inspection and P2-06 pairing
Run cells on Colab CPU. P2-06 reads transcript files from Drive.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os, sys, subprocess
PROJECT_ROOT = Path("/content/drive/MyDrive/PAI&AIES Project")
assert (PROJECT_ROOT / "pyproject.toml").is_file(), "Set PROJECT_ROOT"
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src")
sys.path.insert(0,str(PROJECT_ROOT / "src"))
SNAPSHOT = PROJECT_ROOT / "data/manifests/phase2_raw/20261101T170035Z-7aac9dc154ad"


In [ ]:
import yaml, pyarrow.parquet as pq, json, hashlib, collections, uuid, datetime
from echoproof.contracts import Recording
manifest = json.loads((SNAPSHOT / "snapshot_manifest.json").read_text())
with (SNAPSHOT / "raw_recordings.parquet").open("rb") as f: actual = hashlib.file_digest(f,"sha256").hexdigest()
assert actual == manifest["artifacts_sha256"]["raw_recordings.parquet"]
pqfile = pq.ParquetFile(SNAPSHOT / "raw_recordings.parquet")
assert pqfile.schema_arrow.names == ["schema_name", "schema_version", "payload_json"]
counts = collections.Counter(); ids = set()
for batch in pqfile.iter_batches(batch_size=2048):
    for row in batch.to_pylist():
        assert row["schema_name"] == "recording" and row["schema_version"] == 1
        record = Recording.from_json(row["payload_json"])
        assert record.recording_id not in ids
        ids.add(record.recording_id)
        assert not record.retained and not record.label_verified and not record.method_verified
        counts[(record.generator_family or "genuine", record.label, record.method)] += 1
assert len(ids) == manifest["counts"]["recordings"]
report = {"status":"passed", "parquet_sha256":actual, "record_count":len(ids), "schema":str(pqfile.schema_arrow), "counts":[{"family":k[0],"label":k[1],"method":k[2],"count":v} for k,v in sorted(counts.items())], "all_pending":True}
out = PROJECT_ROOT / "reports/phase2" / ("p2_05_parquet-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8])
out.mkdir(parents=True,exist_ok=False)
(out / "inspection.json").write_text(json.dumps(report,indent=2)+"\n")
print("Inspection report:",out / "inspection.json", json.dumps(report,indent=2))


In [ ]:
check=subprocess.run([sys.executable,"-m","unittest","discover","-s",str(PROJECT_ROOT/"tests"),"-p","test_p2_06_pairing.py","-v"],cwd=PROJECT_ROOT)
assert check.returncode == 0, "Fixture checks failed"


In [ ]:
cmd=[sys.executable,"-m","echoproof.cli","data","pair","--source","phonemedf","--project-root",str(PROJECT_ROOT),"--manifest",str(SNAPSHOT)]
result=subprocess.run(cmd,cwd=PROJECT_ROOT)
assert result.returncode == 0, "Review the pairing report or error above"
